# 03 - MLP Neural Network (with early stopping)

Two-hidden-layer (128, 64) MLP. The non-regularised
version of this exact model (reproducible with
`python train_mlp.py --no-early-stopping`) pushed training
loss down to about 0.024 in 53 epochs while test accuracy
dropped FAR BELOW Logistic Regression - a textbook
overfitting signature. In this notebook we train the
regularised version with early stopping based on
validation (`early_stopping=True`,
`validation_fraction=0.15`, `n_iter_no_change=10`), and
test performance recovers PAST the linear baseline
(see research.md section 3.6).

## 1 - Setup

Here we set matplotlib to inline mode, add the project root
to sys.path so we can import from src/, and bring in the
shared helpers together with the project-wide RANDOM_STATE = 42
(bound once in src/_constants.py, re-exported via src).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# notebooks/ sits one level under the project root, so we add
# the project root to make `from src import ...` work.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import (
    RANDOM_STATE,
    DATASETS,
    load_preprocessed,
    fit_and_score,
    build_metrics_payload,
    save_metrics,
    save_predictions,
    save_test_index,
    save_figure,
    confusion_matrix_figure,
    roc_curve_figure,
    print_dataset_block,
    print_delta,
    model_results_dir,
)

print(f"RANDOM_STATE = {RANDOM_STATE} (bound once in src/_constants.py)")

## 2 - Load the preprocessed feature matrices

load_preprocessed() gives us back the two matrices that we
made in Phase 1: the Baseline (just nutrition + tags) and
the Advanced (the Baseline plus 9 engineered culinary
features). They share the same train/test split so the A/B
comparison is fair.

In [ ]:
datasets, y_train, y_test = load_preprocessed()

print(f"Baseline X_train: {datasets['Baseline'][0].shape}")
print(f"Advanced X_train: {datasets['Advanced'][0].shape}")
print(f"y_test class balance: Miss={int((y_test == 0).sum())} / "
      f"Hit={int((y_test == 1).sum())} "
      f"(majority-class rate: {max(y_test.mean(), 1 - y_test.mean()):.4f})")

## 3 - Configure the model

Validation-based early stopping holds out 15% of X_train
internally (NOT our test split - so it's leakage-free),
tracks the validation accuracy each epoch, and restores
the weights to the best-validation epoch when fit()
finishes. verbose=True gives us the per-epoch loss /
validation log.

In [ ]:
from sklearn.neural_network import MLPClassifier

MODEL_SLUG    = "mlp"
MODEL_NAME    = "MLP (128,64)"
DISPLAY_NAME  = "MLP (128, 64) + early stopping"

MODEL_CONFIG = {
    "hidden_layer_sizes":  (128, 64),
    "max_iter":            300,
    "early_stopping":      True,
    "validation_fraction": 0.15,
    "n_iter_no_change":    10,
    "verbose":             True,
    "random_state":        RANDOM_STATE,
}
MODEL_CONFIG

## 4 - Train on both matrices

For each matrix we build a fresh model from scratch (the
factory below gets called once per dataset). Nothing leaks
from Baseline to Advanced so the delta we see is really
only because of the 9 engineered features.

In [ ]:
def _build_model():
    return MLPClassifier(**MODEL_CONFIG)

per_ds_results = {}
for ds_name in DATASETS:
    X_train, X_test = datasets[ds_name]
    model = _build_model()
    result = fit_and_score(model, X_train, y_train, X_test, y_test)
    per_ds_results[ds_name] = result

    print_dataset_block(ds_name, X_train.shape, result)
    save_predictions(MODEL_SLUG, ds_name, result["y_pred"])

print_delta(per_ds_results)
save_test_index(MODEL_SLUG, y_test)

## 5 - Confusion matrix (Advanced fit)

An annotated heatmap of the confusion matrix from the
Advanced fit. We render it inline and also save it to
results/<slug>/confusion_matrix.png.

In [ ]:
adv = per_ds_results["Advanced"]
cm_array = np.array([
    [adv["confusion_matrix"]["tn"], adv["confusion_matrix"]["fp"]],
    [adv["confusion_matrix"]["fn"], adv["confusion_matrix"]["tp"]],
])
fig_cm = confusion_matrix_figure(
    cm_array,
    title=f"{DISPLAY_NAME} - Confusion Matrix (Advanced)",
)
save_figure(MODEL_SLUG, "confusion_matrix.png", fig_cm)
plt.show()

## 6 - ROC curve + AUC (Advanced fit)

We use predict_proba (or decision_function if the model
doesn't have probabilities). The AUC tells us how well the
model ranks the positives over the negatives, regardless of
the threshold we pick. Phase 4's threshold sweep is built
on top of this.

In [ ]:
fig_roc, auc = roc_curve_figure(
    y_test, adv["proba_hit"],
    title=f"{DISPLAY_NAME} - ROC Curve (Advanced)",
    model_label=DISPLAY_NAME,
)
save_figure(MODEL_SLUG, "roc_curve.png", fig_roc)
print(f"Test ROC AUC (Advanced): {auc:.4f}")
plt.show()

## 7 - Training-loss / validation-error overlay

The training loss (orange) goes smoothly down towards zero,
and the validation error (green) flattens out and starts
climbing a bit - the classic overfitting picture. The
dashed vertical line is the epoch where sklearn restored
the weights back to (the one with the best validation
score).

In [ ]:
adv_mlp = per_ds_results["Advanced"]["model"]
loss = np.asarray(adv_mlp.loss_curve_)
val_scores = np.asarray(adv_mlp.validation_scores_)
val_err = 1.0 - val_scores
epochs = np.arange(1, len(loss) + 1)
best_epoch = int(np.argmax(val_scores)) + 1
best_val_acc = float(val_scores.max())

fig_loss, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, loss, lw=2.0, color="darkorange",
        label="Training loss (log-loss)")
ax.plot(epochs, val_err, lw=2.0, color="seagreen",
        label="Validation error (1 - accuracy)")
ax.axvline(x=best_epoch, color="dimgray", linestyle="--", lw=1.5,
           label=f"Restored epoch = {best_epoch} (best val acc = {best_val_acc:.4f})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss / Error")
ax.set_title(f"MLP - Training Loss vs Validation Error ({len(loss)} epochs)")
ax.grid(alpha=0.3)
ax.legend(loc="best")
fig_loss.tight_layout()
save_figure(MODEL_SLUG, "loss_curve.png", fig_loss)
plt.show()

diagnostics = {
    "epochs_total":             len(loss),
    "best_validation_epoch":    best_epoch,
    "best_validation_accuracy": best_val_acc,
    "training_loss_at_best":    float(loss[best_epoch - 1]),
    "training_loss_at_final":   float(loss[-1]),
    "validation_fraction":      MODEL_CONFIG["validation_fraction"],
    "n_iter_no_change":         MODEL_CONFIG["n_iter_no_change"],
    "loss_curve_plot":          "loss_curve.png",
}
display(pd.Series(diagnostics).to_frame("value"))

## 7.5 - Early-stopping ablation (vs the saved overfit run)

The overfit baseline lives on disk in
results/mlp_overfit/metrics.json, written by
`python train_mlp.py --no-early-stopping` - no hardcoded
numbers. Without early stopping the same architecture runs
its training loss to about 0.024 over 53 epochs and lands
FAR below LR on the test set; with early stopping the
network restores to the best-validation epoch and clears
the linear baseline. See research.md section 3.6.

In [ ]:
from src import load_metrics

now = per_ds_results["Advanced"]
try:
    prior = load_metrics("mlp_overfit")
except FileNotFoundError:
    prior = None
    print("No overfit baseline found - run "
          "`python train_mlp.py --no-early-stopping` once to produce it.")

if prior is not None:
    prior_adv = prior["datasets"]["Advanced"]
    d_acc = now["accuracy"] - prior_adv["accuracy"]
    d_f1 = now["f1"] - prior_adv["f1"]

    display(pd.DataFrame({
        "Overfit baseline (no early stopping)": [
            f"{prior['extras'].get('epochs_total', '?')} epochs",
            f"loss -> {prior['extras'].get('training_loss_at_final', float('nan')):.4f}",
            f"{prior_adv['accuracy']:.4f}",
            f"{prior_adv['f1']:.4f}",
        ],
        "Current (early stopping)": [
            f"{diagnostics['epochs_total']} epochs (restored at {best_epoch})",
            f"loss -> {diagnostics['training_loss_at_final']:.4f}",
            f"{now['accuracy']:.4f} ({d_acc:+.4f})",
            f"{now['f1']:.4f} ({d_f1:+.4f})",
        ],
    }, index=["Training", "Final training loss", "Test Acc", "Test F1"]))

## 8 - Persist the canonical metrics JSON

One JSON per model, written into
results/<slug>/metrics.json. The schema is defined in
src.train_utils.build_metrics_payload, and the master
comparison notebook reads from those files.

In [ ]:
extras = {
    "early_stopping_diagnostics": diagnostics,
    "roc_auc_advanced":           auc,
}

payload = build_metrics_payload(
    model_name=MODEL_NAME,
    display_name=DISPLAY_NAME,
    model_config=MODEL_CONFIG,
    n_train=len(y_train),
    n_test=len(y_test),
    random_state=RANDOM_STATE,
    per_dataset_results=per_ds_results,
    extras=extras,
)
metrics_path = save_metrics(MODEL_SLUG, payload)
print(f"Wrote {metrics_path.relative_to(metrics_path.parent.parent.parent)}")

## 9 - Summary

    **Model:** MLP (128, 64) + early stopping

    - **Test Accuracy / F1:** about 0.6126 / 0.6632 - clears the
  linear baseline on both metrics once we turn on early
  stopping (+1.15 pp Acc, +2.27 pp F1 vs LR).
- **Without early stopping** the exact same architecture
  overfit (Acc 0.5728, far below LR) - see the ablation table
  in section 7.5.
- **The lesson:** more capacity is actually a liability if
  we don't add the matching regularisation. research.md
  section 4 Lesson #6.

    To see this model side by side with the other six, run the
    master comparison notebook (`08_Master_Comparison.ipynb`).